In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/softec-26-ml-competition/test/test/main_df_test.csv
/kaggle/input/competitions/softec-26-ml-competition/test/test/drg_df_test.csv
/kaggle/input/competitions/softec-26-ml-competition/test/test/cpt_df_test.csv
/kaggle/input/competitions/softec-26-ml-competition/test/test/icd_df_test.csv
/kaggle/input/competitions/softec-26-ml-competition/test/test/dob_df.csv
/kaggle/input/competitions/softec-26-ml-competition/train/train/main_df_train.csv
/kaggle/input/competitions/softec-26-ml-competition/train/train/cpt_df_train.csv
/kaggle/input/competitions/softec-26-ml-competition/train/train/drg_df_train.csv
/kaggle/input/competitions/softec-26-ml-competition/train/train/icd_df_train.csv
/kaggle/input/competitions/softec-26-ml-competition/train/train/dob_df.csv


**data preprocessing#**

In [5]:
import pandas as pd

# Define the exact paths from your directory
train_path = '/kaggle/input/competitions/softec-26-ml-competition/train/train/'
test_path = '/kaggle/input/competitions/softec-26-ml-competition/test/test/'

# Load main training file
df_main = pd.read_csv(f'{train_path}main_df_train.csv')

In [7]:
df_main.shape

(807420, 301)

In [8]:
df_main.head()

,Member_Key,YEAR,MONTH,ProviderVisitCount,ProviderCharges,TotalCost,MemberHccScore,MemberHccScoreLastYear,MemberHccScoreLastTwoYear,ERAdmisison,...,DeniedCost,AWVProviderNetwork,EDVisitsWithFollowUp,IPVisitsWithFollowUp,TotalScriptsFilled,Admission,AdmissionCost,AdmissionLOS,NEXT_YEAR_COST,HighCostLabel
0,1009876,2022,9,0,0.00,53.50,0.701222,0.0,0.0,0,...,0.5,NaN,0.0,0.0,0,0,0.5,0,89.41,0
1,985607,2022,7,1,114.22,261.80,0.470306,0.0,0.0,0,...,0.5,In-Network,0.0,0.0,0,0,0.5,0,1677.60,0
2,1015510,2021,9,0,0.00,53.50,0.740149,0.0,0.0,0,...,0.5,In-Network,0.0,0.0,0,0,0.5,0,2144.99,0
3,1208517,2021,1,0,0.00,53.50,0.480346,0.0,0.0,0,...,0.5,NaN,0.0,0.0,0,0,0.5,0,699.82,0
4,1039845,2022,10,0,0.00,3419.48,1.306053,0.0,0.0,0,...,0.5,NaN,0.0,0.0,0,0,0.5,0,28045.49,0


In [9]:
df_main.dtypes


Member_Key              int64
YEAR                    int64
MONTH                   int64
ProviderVisitCount      int64
ProviderCharges       float64
                       ...   
Admission               int64
AdmissionCost         float64
AdmissionLOS            int64
NEXT_YEAR_COST        float64
HighCostLabel           int64
Length: 301, dtype: object

In [10]:
df_main.isnull().mean()

Member_Key            0.0
YEAR                  0.0
MONTH                 0.0
ProviderVisitCount    0.0
ProviderCharges       0.0
                     ... 
Admission             0.0
AdmissionCost         0.0
AdmissionLOS          0.0
NEXT_YEAR_COST        0.0
HighCostLabel         0.0
Length: 301, dtype: float64

In [11]:
#1. Row Type Distribution
print("Row types in MONTH column:")
print(df_main['MONTH'].value_counts())

#2. Class Imbalance (The most important check)

print("\nRatio of High-Cost vs Low-Cost:")
print(df_main['HighCostLabel'].value_counts(normalize=True))

#3. Year Verification
print("\nYears present in training data:")
print(df_main['YEAR'].unique())

Row types in MONTH column:
MONTH
 12    64168
 11    64132
 10    64057
 9     63933
 8     63789
 7     63637
 6     63479
 5     63319
 4     63152
 3     62971
 2     62772
 1     62617
-1     45394
Name: count, dtype: int64

Ratio of High-Cost vs Low-Cost:
HighCostLabel
0    0.888965
1    0.111035
Name: proportion, dtype: float64

Years present in training data:
[2022 2021 2023]


**master table**

In [13]:
#base
master_df = df_main[df_main['MONTH'].isin([-1, 12]).copy()]

print(f"Master table base rows: {master_df.shape[0]}")

Master table base rows: 109562


In [17]:
#loading other fles

df_icd = pd.read_csv(f'{train_path}icd_df_train.csv')
df_cpt = pd.read_csv(f'{train_path}cpt_df_train.csv')
df_dob = pd.read_csv(f'{train_path}dob_df.csv')

#aggregate functions

# 1. count unique values
icd_counts = df_icd.groupby(['Member_Key', 'Start_Date'])['Diagnosis_Code'].nunique().reset_index()
icd_counts.columns = ['Member_Key', 'Year',  'unique_icd_count']

# 2. count total procedures
cpt_counts = df_cpt.groupby(['Member_Key', 'StartDate']).size().reset_index()
cpt_counts.columns = ['Member_key', 'YEAR', 'total_cpt_count']

In [20]:
# Check the columns of each dataframe before merging
print("Master columns:", master_df.columns.tolist()[:5]) # Show first 5
print("ICD columns:", icd_counts.columns.tolist())
print("CPT columns:", cpt_counts.columns.tolist())

Master columns: ['Member_Key', 'YEAR', 'MONTH', 'ProviderVisitCount', 'ProviderCharges']
ICD columns: ['Member_Key', 'Year', 'unique_icd_count']
CPT columns: ['Member_key', 'YEAR', 'total_cpt_count']


In [22]:
# Standardize ICD names 
icd_counts.columns = ['Member_Key', 'YEAR', 'unique_icd_count']

# Standardize CPT names
cpt_counts.columns = ['Member_Key', 'YEAR', 'total_cpt_count']

# Merge ICD
master_df = master_df.merge(icd_counts, on=['Member_Key', 'YEAR'], how='left')

# Merge CPT
master_df = master_df.merge(cpt_counts, on=['Member_Key', 'YEAR'], how='left')

# Merge Demographics 
master_df = master_df.merge(df_dob, on='Member_Key', how='left')

# Final Clean-up
master_df['unique_icd_count'] = master_df['unique_icd_count'].fillna(0)
master_df['total_cpt_count'] = master_df['total_cpt_count'].fillna(0)

print("Final master table shape: ", master_df.shape)

Final master table shape:  (109562, 306)


In [23]:
master_df.head()

,Member_Key,YEAR,MONTH,ProviderVisitCount,ProviderCharges,TotalCost,MemberHccScore,MemberHccScoreLastYear,MemberHccScoreLastTwoYear,ERAdmisison,...,Admission,AdmissionCost,AdmissionLOS,NEXT_YEAR_COST,HighCostLabel,unique_icd_count,total_cpt_count,DOB_Key,Gender_Key,AgeGroup_Key
0,1068421,2022,12,0,0.00,200.04,0.401455,0.0,0.0,0,...,0,0.5,0,3894.79,0,17.0,55.0,1945-09-05,1,5
1,1080473,2022,12,0,0.00,90.28,1.939483,0.0,0.0,0,...,0,0.5,0,132388.86,1,24.0,76.0,1947-01-13,1,5
2,1085884,2021,12,0,0.00,155.75,0.741680,0.0,0.0,0,...,0,0.5,0,862.41,0,35.0,34.0,1949-09-28,1,5
3,1117224,2023,12,0,0.00,53.50,0.614997,0.0,0.0,0,...,0,0.5,0,882.26,0,11.0,10.0,1937-02-18,1,5
4,1068449,2023,-1,1,65.08,6648.64,2.010782,0.0,0.0,0,...,0,6.0,0,4978.16,0,37.0,33.0,1961-05-19,2,4


In [8]:
#identifying paths for annoying errors
df_drg = pd.read_csv(f'{train_path}drg_df_train.csv')

# 1. Aggregate DRG
drg_counts = df_drg.groupby(['Member_Key', 'Start_Date_Year']).size().reset_index(name='total_drg_count')
drg_counts.rename(columns={'Start_Date_Year': 'YEAR'}, inplace=True)

# 2. Merging and Finalizing
master_df = master_df.merge(drg_counts, on=['Member_Key', 'YEAR'], how='left')
master_df['total_drg_count'] = master_df['total_drg_count'].fillna(0) 

# Convert dob into datetime
master_df['birth_year'] = pd.to_datetime(master_df['DOB_Key']).dt.year
master_df['age'] = master_df['YEAR'] - master_df['birth_year']

# handle missing values
if 'AWVProviderNetwork' in master_df.columns:
    master_df['AWVProviderNetwork'] = master_df['AWVProviderNetwork'].fillna('Unknown')

**final pipeline**

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Define your paths
TRAIN_PATH = '/kaggle/input/competitions/softec-26-ml-competition/train/train/'
TEST_PATH = '/kaggle/input/competitions/softec-26-ml-competition/test/test/'

def preprocess_healthcare_pipeline(main_path, icd_path, cpt_path, drg_path, dob_path):
    # Load data
    main_df = pd.read_csv(main_path)
    icd_df = pd.read_csv(icd_path)
    cpt_df = pd.read_csv(cpt_path)
    drg_df = pd.read_csv(drg_path)
    dob_df = pd.read_csv(dob_path)

    # 1. THE BASE: Use -1 to avoid duplicates (ensures unique Member_Key + YEAR)
    master = main_df[main_df['MONTH'] == -1].copy()
    
    # 2. ROBUST YEAR EXTRACTION (Prevents the '1970' silent failure)
    def extract_year_safe(df, col):
        return pd.to_numeric(df[col].astype(str).str[:4], errors='coerce')

    # Aggregating ICD
    icd_df['YEAR'] = extract_year_safe(icd_df, 'Start_Date')
    icd_counts = icd_df.groupby(['Member_Key', 'YEAR'])['Diagnosis_Code'].nunique().reset_index(name='unique_icd_count')

    # Aggregating CPT
    cpt_df['YEAR'] = extract_year_safe(cpt_df, 'StartDate')
    cpt_counts = cpt_df.groupby(['Member_Key', 'YEAR']).size().reset_index(name='total_cpt_count')

    # Aggregating DRG
    drg_df['YEAR'] = extract_year_safe(drg_df, 'Start_Date_Year')
    drg_counts = drg_df.groupby(['Member_Key', 'YEAR']).size().reset_index(name='total_drg_count')

    # 3. STANDARDIZE KEYS (Ensures perfect merge matching)
    for df_tmp in [master, icd_counts, cpt_counts, drg_counts]:
        df_tmp['Member_Key'] = df_tmp['Member_Key'].astype(int)
        df_tmp['YEAR'] = df_tmp['YEAR'].astype(int)

    # 4. RELATION MERGE
    master = master.merge(icd_counts, on=['Member_Key', 'YEAR'], how='left')
    master = master.merge(cpt_counts, on=['Member_Key', 'YEAR'], how='left')
    master = master.merge(drg_counts, on=['Member_Key', 'YEAR'], how='left')
    master = master.merge(dob_df, on='Member_Key', how='left')

    # 5. DATA CLEANING (Defining the missing list here!)
    count_cols = ['unique_icd_count', 'total_cpt_count', 'total_drg_count']
    master[count_cols] = master[count_cols].fillna(0)

    # 6. LAG FEATURES (Predictive Power)
    master = master.sort_values(['Member_Key', 'YEAR'])
    master['TotalCost_lag1'] = master.groupby('Member_Key')['TotalCost'].shift(1)
    master['cost_trend'] = master['TotalCost'] - master['TotalCost_lag1']

    # 7. AGE CALCULATION
    if 'DOB_Key' in master.columns:
        master['birth_year'] = pd.to_numeric(master['DOB_Key'].astype(str).str[:4], errors='coerce')
        master['age'] = master['YEAR'] - master['birth_year']
    else:
        master['age'] = None  # Handle missing DOB_Key

    # Handle categorical NaNs
    if 'AWVProviderNetwork' in master.columns:
        master['AWVProviderNetwork'] = master['AWVProviderNetwork'].fillna('Unknown')

    # Drop temporary columns
    master = master.drop(columns=['birth_year', 'DOB_Key'], errors='ignore')

    return master


# --- RUNNING THE PIPELINE ---

# 1. Process Training Data (Years 2021-2023)
print("Processing Training Table...")
master_train = preprocess_healthcare_pipeline(
    f'{TRAIN_PATH}main_df_train.csv',
    f'{TRAIN_PATH}icd_df_train.csv',
    f'{TRAIN_PATH}cpt_df_train.csv',
    f'{TRAIN_PATH}drg_df_train.csv',
    f'{TRAIN_PATH}dob_df.csv'
)

# 2. Process Test Data (Year 2024)
print("Processing Test Table...")
master_test = preprocess_healthcare_pipeline(
    f'{TEST_PATH}main_df_test.csv',
    f'{TEST_PATH}icd_df_test.csv',
    f'{TEST_PATH}cpt_df_test.csv',
    f'{TEST_PATH}drg_df_test.csv',
    f'{TEST_PATH}dob_df.csv'
)

# 3. Final Polish: Scaling
cols_to_scale = ['age', 'total_cpt_count', 'unique_icd_count', 'total_drg_count']
master_train[cols_to_scale] = scaler.fit_transform(master_train[cols_to_scale])
master_test[cols_to_scale] = scaler.transform(master_test[cols_to_scale])

# 4. EXPORT FOR TEAMMATES
master_train.to_csv('final_master_train.csv', index=False)
master_test.to_csv('final_master_test.csv', index=False)

print("Done! Files 'final_master_train.csv' and 'final_master_test.csv' are ready.")
# Check the Year range - should be 2021, 2022, 2023

Processing Training Table...
Processing Test Table...
Done! Files 'final_master_train.csv' and 'final_master_test.csv' are ready.


**checking the pipepline**

In [ ]:
# Did the join find matches? 
# If 'inner' is close to your master size, it's a success!
print(f"Master size: {len(master_train)}")
print(f"Rows with at least one ICD code: {len(master_train[master_train['unique_icd_count'] > 0])}")

print(f"Years in ICD counts: {icd_counts['YEAR'].unique()}")

In [36]:
master_train.head()

,Member_Key,YEAR,MONTH,ProviderVisitCount,ProviderCharges,TotalCost,MemberHccScore,MemberHccScoreLastYear,MemberHccScoreLastTwoYear,ERAdmisison,...,AdmissionCost,AdmissionLOS,NEXT_YEAR_COST,HighCostLabel,unique_icd_count,total_cpt_count,total_drg_count,Gender_Key,AgeGroup_Key,age
0,1068421,2022,12,0,0.00,200.04,0.401455,0.0,0.0,0,...,0.5,0,3894.79,0,-0.772342,-0.228333,-0.478896,1,5,0.404147
1,1080473,2022,12,0,0.00,90.28,1.939483,0.0,0.0,0,...,0.5,0,132388.86,1,-0.532000,-0.084314,-0.238315,1,5,0.214764
2,1085884,2021,12,0,0.00,155.75,0.741680,0.0,0.0,0,...,0.5,0,862.41,0,-0.154318,-0.372352,-0.639284,1,5,-0.069310
3,1117224,2023,12,0,0.00,53.50,0.614997,0.0,0.0,0,...,0.5,0,882.26,0,-0.978350,-0.536946,-0.639284,1,5,1.256370
4,1068449,2023,-1,1,65.08,6648.64,2.010782,0.0,0.0,0,...,6.0,0,4978.16,0,-0.085649,-0.379210,-0.318509,2,4,-1.016225


In [1]:
print(master_train['unique_icd_count'].isnull().mean())  # should be low, not 0.8+
print(master_train['total_cpt_count'].describe())

NameError: name 'master_train' is not defined

In [7]:
print(master_train['unique_icd_count'].isnull().mean())   # should be under 0.3
print(master_train['total_cpt_count'].isnull().mean())    # same

0.0
0.0


In [8]:
dupes = master_train.groupby(['Member_Key', 'YEAR']).size()
print(dupes[dupes > 1].shape[0])   # should print 0

0


In [16]:
print(master_train['unique_icd_count'].describe())   # should show non-zero mean

count    4.539400e+04
mean     1.001779e-17
std      1.000011e+00
min     -1.354765e+00
25%     -6.998957e-01
50%     -2.518271e-01
75%      4.030425e-01
max      9.261015e+00
Name: unique_icd_count, dtype: float64


In [10]:
print(master_train['HighCostLabel'].isnull().mean())  # should be 0 or very close

0.0


**Adding model**

In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# --- LOAD YOUR CLEAN DATA ---
train = pd.read_csv('final_master_train.csv')
test  = pd.read_csv('final_master_test.csv')

# --- DEFINE FEATURES AND LABEL ---
drop_cols = ['HighCostLabel', 'Member_Key', 'NEXT_YEAR_COST', 
             'MONTH', 'AWVProviderNetwork']

X = train.drop(columns=[c for c in drop_cols if c in train.columns])
y = train['HighCostLabel'].astype(int)

X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

# Drop any remaining object/string columns — XGBoost can't handle them
object_cols = X.select_dtypes(include='object').columns.tolist()
X = X.drop(columns=object_cols)
X_test = X_test.drop(columns=[c for c in object_cols if c in X_test.columns])

# Align columns — test must match train exactly
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# --- CLASS IMBALANCE RATIO ---
neg = (y == 0).sum()
pos = (y == 1).sum()
ratio = neg / pos
print(f"Class ratio (scale_pos_weight): {ratio:.2f}")

# --- CROSS VALIDATION TO FIND BEST THRESHOLD ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(y))

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=ratio,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)

print("Running cross-validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              verbose=False)
    
    oof_probs[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_f1 = f1_score(y_val, oof_probs[val_idx] >= 0.5)
    print(f"  Fold {fold+1} F1 @ 0.5: {fold_f1:.4f}")

# --- FIND BEST THRESHOLD ON OOF PREDICTIONS ---
print("\nSearching for best threshold...")
thresholds = np.arange(0.1, 0.6, 0.01)
best_t = max(thresholds, key=lambda t: f1_score(y, oof_probs >= t))
best_f1 = f1_score(y, oof_probs >= best_t)
print(f"Best threshold: {best_t:.2f}  |  OOF F1: {best_f1:.4f}")
print(classification_report(y, oof_probs >= best_t))


# --- RETRAIN ON FULL DATA, PREDICT TEST ---
print("\nRetraining on full data...")
model.fit(X, y, verbose=False)
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_t).astype(int)

# --- BUILD SUBMISSION FILE ---
submission = pd.DataFrame({
    'Member_Key': test['Member_Key'],
    'HighCostLabel': test_preds
})
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission ready. Predicted high-cost members: {test_preds.sum()}")
print(submission['HighCostLabel'].value_counts())

Class ratio (scale_pos_weight): 7.82
Running cross-validation...
  Fold 1 F1 @ 0.5: 0.4352
  Fold 2 F1 @ 0.5: 0.4056
  Fold 3 F1 @ 0.5: 0.4160
  Fold 4 F1 @ 0.5: 0.4104
  Fold 5 F1 @ 0.5: 0.3887

Searching for best threshold...
Best threshold: 0.59  |  OOF F1: 0.4162
              precision    recall  f1-score   support

           0       0.93      0.92      0.92     40250
           1       0.41      0.43      0.42      5144

    accuracy                           0.86     45394
   macro avg       0.67      0.67      0.67     45394
weighted avg       0.87      0.86      0.87     45394


Retraining on full data...

Submission ready. Predicted high-cost members: 1750
HighCostLabel
0    10637
1     1750
Name: count, dtype: int64


In [6]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

train = pd.read_csv('final_master_train.csv')
test  = pd.read_csv('final_master_test.csv')

# -------------------------------------------------------
# STEP A — prepare features
# drop anything that would leak the answer or isn't numeric
# -------------------------------------------------------
drop_cols = ['HighCostLabel', 'Member_Key', 'NEXT_YEAR_COST',
             'MONTH', 'AWVProviderNetwork']

X = train.drop(columns=[c for c in drop_cols if c in train.columns])
y = train['HighCostLabel'].astype(int)

X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

# remove any leftover string columns — tree models can't use them
object_cols = X.select_dtypes(include='object').columns.tolist()
X     = X.drop(columns=object_cols)
X_test = X_test.drop(columns=[c for c in object_cols if c in X_test.columns])
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# --- SAFETY CHECK: Drop constant columns ---
# If every row has the same value, the model learns nothing and the scaler breaks.
constant_cols = [c for c in X.columns if X[c].nunique() <= 1]
print(f"Dropping {len(constant_cols)} constant columns: {constant_cols}")
X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=[c for c in constant_cols if c in X_test.columns])

# Final NaN fill just in case anything weird happened during engineering
X = X.fillna(0)
X_test = X_test.fillna(0)

# -------------------------------------------------------
# STEP B — understand the imbalance before touching models
# -------------------------------------------------------
neg = (y == 0).sum()
pos = (y == 1).sum()
ratio = neg / pos
print(f"Low-cost members : {neg}")
print(f"High-cost members: {pos}")
print(f"Imbalance ratio  : {ratio:.1f}x")
# this ratio goes directly into XGBoost as scale_pos_weight
# it tells the model "a miss on the minority class costs X times more"

# -------------------------------------------------------
# STEP C — Revised Logistic Regression
# -------------------------------------------------------
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr__oof = np.zeros(len(y))

# We move scaling INSIDE the loop to be more professional (prevents leakage)
for train_idx, val_idx in skf.split(X, y):
    # Split the raw data
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # Fit scaler ONLY on training fold
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)
    
    # Fit model
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, y_train_fold)
    
    # Predict probabilities
    lr__oof[val_idx] = lr.predict_proba(X_val_scaled)[:, 1]

# Calculate F1 score using a standard 0.5 threshold
lr_f1 = f1_score(y, lr__oof >= 0.5)
print(f"\nLogistic Regression baseline F1: {lr_f1:.4f}")
# -------------------------------------------------------
# STEP D — move to XGBoost and find the right threshold
# random search over a small grid, not full Optuna
# keeps it readable and explainable
# -------------------------------------------------------
param_grid = [
    {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 300},
    {'max_depth': 6, 'learning_rate': 0.05, 'n_estimators': 500},
    {'max_depth': 6, 'learning_rate': 0.01, 'n_estimators': 700},
]

best_f1    = 0
best_params = None
best_oof   = None

print("\nSearching over parameter combinations...")
for params in param_grid:
    oof = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X, y):
        m = XGBClassifier(
            **params,
            scale_pos_weight=ratio,
            eval_metric='aucpr',
            random_state=42,
            n_jobs=-1
        )
        m.fit(X.iloc[train_idx], y.iloc[train_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              verbose=False)
        oof[val_idx] = m.predict_proba(X.iloc[val_idx])[:, 1]

    # find best threshold for this param combo
    thresholds = np.arange(0.10, 0.55, 0.01)
    t = max(thresholds, key=lambda t: f1_score(y, oof >= t))
    f1 = f1_score(y, oof >= t)
    print(f"  params={params} | best_threshold={t:.2f} | F1={f1:.4f}")

    if f1 > best_f1:
        best_f1     = f1
        best_params = params
        best_oof    = oof
        best_t      = t

print(f"\nBest config : {best_params}")
print(f"Best threshold: {best_t:.2f}")
print(f"Best OOF F1   : {best_f1:.4f}")
print("\nFull report on OOF predictions:")
print(classification_report(y, best_oof >= best_t))

# -------------------------------------------------------
# STEP E — retrain on full data with best config, predict
# -------------------------------------------------------
final_model = XGBClassifier(
    **best_params,
    scale_pos_weight=ratio,
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1
)
final_model.fit(X, y, verbose=False)
test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_t).astype(int)

# -------------------------------------------------------
# STEP F — quick check before submitting
# if predicted high-cost count is under 300, threshold too high
# if it's over 5000, threshold too low
# -------------------------------------------------------
print(f"\nPredicted high-cost members in test: {test_preds.sum()}")
print(f"That's {test_preds.mean()*100:.1f}% of test set")

submission = pd.DataFrame({
    'Member_Key': test['Member_Key'],
    'HighCostLabel': test_preds
})
submission.to_csv('submission.csv', index=False)
print("submission.csv is ready")

Dropping 77 constant columns: ['MemberHccScoreLastYear', 'MemberHccScoreLastTwoYear', 'ERAdmisison', '30dayHospitalReadmission', 'ChronicObstructivePulmonaryDiseaseOrAsthma', 'CongestiveHeartFailure', 'BacterialPneumonia', 'DiabetesLongTermComplications', 'AmputationDiabetes', 'ERAdmisison_Cost', '30dayHospitalReadmission_Cost', 'EmergencyDepartmentVisitsWithAdmissions_Cost', 'ChronicObstructivePulmonaryDiseaseOrAsthma_Cost', 'CongestiveHeartFailure_Cost', 'BacterialPneumonia_Cost', 'AmputationDiabetes_Cost', 'Medication_Cost', 'IsLatest', 'AWV', 'EDVisitsWithNoFollowUp', 'QUARTER', 'ActualYear', 'ActualMonthNumber', 'PercentageReadmission', 'PersiviaMemberHccScore', 'Payers_key', 'DentistEvents', 'DentistCost', 'ElderlyWaiverEvents', 'ElderlyWaiverCost', 'IPPE_Eligible', 'IPPE_Compliant', 'IsCovid', 'NonClaimBasedPayment', 'PredictedCost', 'DMEPhysicianPrefferedSpend', 'DMEPOSPrefferedSpend', 'NonDMEPOSPrefferedSpend', 'InpatientPrefferedSpend', 'HHAPrefferedSpend', 'HospicePrefferedS

In [7]:
# Check the first few rows to ensure column names and types are correct
sub_check = pd.read_csv('submission.csv')
print(sub_check.head())
print(f"\nTotal rows in submission: {len(sub_check)}")

   Member_Key  HighCostLabel
0      985602              0
1      985603              0
2      985607              0
3      985619              0
4      985644              0

Total rows in submission: 12387


In [8]:
print(sub_check['HighCostLabel'].value_counts())

HighCostLabel
0    9816
1    2571
Name: count, dtype: int64
